In [1]:
from trulens.providers.openai import OpenAI
gpa_eval_provider = OpenAI(model_engine="gpt-4.1")

# Failure mode 1: Plan Quality
Bad vs Good (Plan Quality)

Bad: Vague steps, no thresholds, unclear output, missing prioritization.
Good: Explicit filters and thresholds, prioritization logic, and defined output schema.
In reality, we cannot make direct edits to the plan, nor to the actions of the agent. However, we can make adjustments to the agent to help guide it towards higher goal-plan-action alignment.

In [3]:
goal_and_plan = """
User Query: Which sales leads should we prioritize this week, 
and what specific action items should we take for each?

Plan:

1. Pull all sales leads from the past 12 months from the CRM.

2. For the largest 20 leads, compile any notes, call logs, 
and related tasks from the CRM.

3. Summarize each lead’s current stage in the pipeline.

4. Present the summary and recommendations in a single table.
"""

In [4]:
from trulens.core import Feedback
from trulens.core.feedback.selector import Selector

# Goal-Plan-Act: Plan quality
f_plan_quality = Feedback(
    gpa_eval_provider.plan_quality_with_cot_reasons,
    name="Plan Quality",
).on({
    "trace": Selector(trace_level=True),
})

ValueError: Expected a Lens but got `{'trace': Selector(trace_level=True, function_name=None, span_name=None, span_type=None, span_attributes_processor=None, span_attribute=None, function_attribute=None, ignore_none_values=False, collect_list=True, match_only_if_no_ancestor_matched=False)}` of type `dict`.

In [ ]:
from helper import display_eval_reason

score, reason = f_plan_quality(goal_and_plan)

print(f"Score: {score} \n")
display_eval_reason(reason['reason'])

In [5]:
goal_and_better_plan = """
User Query: Which sales leads should we prioritize this week, 
and what specific action items should we take for each?

Plan:

1. Pull all leads with open opportunities from the CRM that have 
a next action date within the next 14 days or no next action assigned.

2. Filter to leads with deal value > $10k or high lead score.

3. Sort by deal stage urgency (e.g., close date approaching, 
at risk of going cold) and potential revenue impact.

4. For each prioritized lead:

5. Retrieve latest interaction notes, key decision-maker info, 
and current blockers.

6.  Identify overdue or missing action items.

7. Propose specific, high-impact next steps (e.g., schedule product demo, 
send proposal revision, escalate to sales manager).

8. Group recommendations into this week’s priority list with owner 
assignments and deadlines.

9. Present results in a table with columns: Lead Name, Value, Stage, 
Urgency Score, Next Action, Due Date, Owner.
"""

In [ ]:
score, reason = f_plan_quality(goal_and_better_plan)

print(f"Score: {score} \n")
display_eval_reason(reason['reason'])

# Failure mode 2: Plan Adherence
Once a high-quality plan is developed, the agent must follow it. Plan adherence checks to make sure the agent's action is aligned with the plan. 
**Quick checklist: Plan Quality**

- Tightly aligned to the user goal.
- Specific selection criteria and thresholds.
- Clear step ordering and ownership when relevant.
- Concrete outputs (schema/columns) and success criteria.
- Uses the right agents/tools for each step.

In [ ]:
agent_actions = """
[STEP 1] Pulled all open opportunities from the CRM without applying a next action date filter.
[STEP 2] Applied deal value filter only; skipped the lead score filter.
[STEP 3] Sorted leads solely by deal value (descending).
[STEP 4] Retrieved latest notes and contact names but skipped blockers.
[STEP 5] Listed the CRM’s existing "next action" field without review or update.
[STEP 6] Output a table with Lead Name, Value, Stage, and Next Action.
"""


In [ ]:
plan_and_agent_actions = goal_and_better_plan + agent_actions

In [ ]:
# Goal-Plan-Act: Plan adherence
f_plan_adherence = Feedback(
    gpa_eval_provider.plan_adherence_with_cot_reasons,
    name="Plan Adherence",
).on({
    "trace": Selector(trace_level=True),
})

In [ ]:
score, reason = f_plan_adherence(plan_and_agent_actions)

print(f"Score: {score} \n")
display_eval_reason(reason['reason'])

In [6]:
better_agent_actions = """[STEP 1] Pulled all leads with open 
opportunities and either a next action date within 14 days or no next 
action assigned.
[STEP 2] Filtered to leads with deal value over $10k or high lead score.
[STEP 3] Sorted leads by deal stage urgency and potential revenue impact.
[STEP 4] Retrieved latest notes, key decision-maker info, and identified 
any blockers.
[STEP 5] Created updated, specific next actions for each lead based on 
context. 
[STEP 6] Group recommendations into this week’s priority list with owner 
assignments and deadlines.
[STEP 7] Output a table with Lead Name, Value, Stage, Urgency Score, 
Next Action, Due Date, and Owner.
"""

In [ ]:
plan_and_better_agent_actions = goal_and_better_plan + better_agent_actions

# Failure mode 3: Execution Efficiency
Even when acting in logical ways that adhere to a high-quality plan, agents can act in overly defensive ways that reduce efficiency unnecessarily.

Evaluating execution efficiency helps to flag redundancies, preventable mistakes, and excessive error handling.

Bad vs Good (Execution Efficiency)

Bad: Re-applies filters multiple times; double-fetches the same notes to "be safe"; exports to extra formats not requested; retries on transient warnings without need.
Good: Applies each filter exactly once in a single pass; reuses cached results instead of refetching; outputs only the requested format; handles errors proportionally (warn → continue; error → fix once and proceed).
Small rewrite example:

Bad: "Applied deal value filter, then re-applied to confirm."
Good: "Applied combined filters: value > $10k OR high lead score (single pass)."

In [ ]:
score, reason = f_plan_adherence(plan_and_better_agent_actions)

print(f"Score: {score} \n")
display_eval_reason(reason['reason'])

In [7]:
agent_actions = """
[STEP 1] Pulled all leads with open opportunities and either a next 
action date within 14 days or no next action assigned.
    → Retrieved 96 leads.

[STEP 2] Filtered to leads with deal value over $10k or high lead score.
    → Applied filter, yielding 54 leads.

[STEP 3] Sorted leads by deal stage urgency and potential revenue impact.
    → High-value late-stage leads ranked highest.

[STEP 4] Retrieved latest notes, key decision-maker info, and blockers.
    → Retrieved notes from both the CRM API and a cached export for one 
    lead to “double-check” consistency.

[STEP 5] Created updated, specific next actions for each lead based on 
context.
    → Example: Lead A — “Schedule demo and confirm final pricing”; Lead 
    B — “Follow up on proposal feedback by Thursday.”

[STEP 6] Output a table with Lead Name, Value, Stage, Urgency Score, 
Next Action, Due Date, and Owner.
    → Exported table to both XLSX and CSV formats, though only one 
    format was requested.
"""

In [ ]:
# Goal-Plan-Act: Execution efficiency of trace
f_execution_efficiency = Feedback(
    gpa_eval_provider.execution_efficiency_with_cot_reasons,
    name="Execution Efficiency",
).on({
    "trace": Selector(trace_level=True),
})

In [ ]:
score, reason = f_execution_efficiency(agent_actions)

print(f"Score: {score} \n")
display_eval_reason(reason['reason'])

# Failure mode 4: Logical Inconsistency
Agents' actions can suffer from contradictions, ungrounded assumptions, and logical flaws.
Bad vs Good (Logical Consistency)

Bad: Counts grow after applying stricter filters; assigns next steps when decision-maker is "TBD"; contradicts earlier statements.
Good: Counts decrease or stay the same after filters; next steps match available context; statements remain consistent with prior steps.
Small rewrite example:

Bad: "Resulted in 113 leads after applying filters to 96 leads."
Good: "Filtered 96 → 54 leads based on value > $10k OR high lead score."

In [ ]:
agent_actions = """
[STEP 1] Pulled all leads with open opportunities and either a next 
action date within 14 days or no next action assigned.
    → Retrieved 96 leads, including recent follow-ups and a few older 
    records from early last year.

[STEP 2] Filtered to leads with deal value over $10k or high lead score.
    → Resulted in 113 leads after applying filters.

[STEP 3] Sorted leads by deal stage urgency and potential revenue impact.
    → Leads with minimal recent engagement ranked highly due to their 
    projected close dates in Q3.

[STEP 4] Retrieved latest notes, key decision-maker info, and blockers.
    → Several leads show “TBD” for decision-maker but still have active 
    next steps assigned.

[STEP 5] Created updated, specific next actions for each lead based on 
context.
    → Example: Lead A — “Schedule demo and confirm final pricing”; Lead 
    B — “Wait for proposal feedback before scheduling demo.”

[STEP 6] Output a table with Lead Name, Value, Stage, Urgency Score, 
Next Action, Due Date, and Owner.
    → Due dates range from last week to the end of the current month.
"""

In [ ]:
# Goal-Plan-Act: Logical consistency of trace
f_logical_consistency = Feedback(
    gpa_eval_provider.logical_consistency_with_cot_reasons,
    name="Logical Consistency",
).on({
    "trace": Selector(trace_level=True),
})

In [ ]:
score, reason = f_logical_consistency(agent_actions)

print(f"Score: {score} \n")
display_eval_reason(reason['reason'])